# SSIF_V3：Google Colab 中文實作教學

研究資料：

```python
data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
```

此 CSV 為「每列一個完整地震事件」格式，巢狀欄位可能遠大於 Python `csv` 預設的 128 KiB。本版在任何 CSV 讀取前先提高欄位上限，並以 streaming 方式掃描與轉換。

In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 安全同步 GitHub repository

In [10]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_ROOT = Path('/content/SSIF_V3')
REPO_URL = 'https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode:
        raise RuntimeError(
            f"exit {result.returncode}: " + " ".join(map(str, command))
        )
    return result

# 必須先離開可能被更新或刪除的 repository 目錄。
os.chdir('/content')

if (REPO_ROOT / '.git').is_dir():
    try:
        run_checked(['git', '-C', str(REPO_ROOT), 'fetch', '--prune', 'origin'])
        run_checked(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'])
        run_checked(['git', '-C', str(REPO_ROOT), 'clean', '-fd'])
    except RuntimeError:
        os.chdir('/content')
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])
else:
    os.chdir('/content')
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])

sha = run_checked(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD']
).stdout.strip()
print('Repository commit:', sha)

run_checked([
    'python', '-m', 'pip', 'install', '-q',
    '-r', str(REPO_ROOT / 'requirements.txt')
])


HEAD is now at 3b3ce64 Require large CSV field limit in Colab validation
3b3ce645e4a9d8ab420c31d1d5dd448ddd532f07
Repository commit: 3b3ce645e4a9d8ab420c31d1d5dd448ddd532f07


CompletedProcess(args=['python', '-m', 'pip', 'install', '-q', '-r', '/content/SSIF_V3/requirements.txt'], returncode=0, stdout='', stderr='')

## 2. 路徑、CSV 大欄位設定與執行開關

In [11]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

# combined_data.csv 的 intensity/stids 等欄位可超過 131072 bytes。
_csv_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(_csv_limit)
        break
    except OverflowError:
        _csv_limit //= 10

assert csv.field_size_limit() > 131072
print('CSV field limit:', csv.field_size_limit())

data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
DATA_CSV = Path(data_path)
WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')

TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'
REPORT_DIR = WORK_ROOT / 'reports'
PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
MODEL_DIR = WORK_ROOT / 'models' / 'seed_20260728'

SCAN_REPORT = REPORT_DIR / 'combined_data_scan.json'
VALIDATION_REPORT = REPORT_DIR / 'converted_archive_validation.json'

WINDOWS = [10, 15, 20, 25, 30, 35, 40]
SEED = 20260728
DUPLICATE_POLICY = 'skip-identical'

RUN_FULL_SCAN = False
RUN_FULL_CONVERSION = False
RUN_FULL_VALIDATION = False
RUN_AUDIT_SPLIT = False
RUN_QUICK_TRAIN = False
RUN_FULL_TRAIN = False

for path in [TRAIN_DATA, REPORT_DIR, PREPARED_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.is_file(), f'找不到研究資料：{DATA_CSV}'
print('CSV size (GB):', round(DATA_CSV.stat().st_size / 1024**3, 3))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CSV field limit: 9223372036854775807
CSV size (GB): 0.678
GPU: NVIDIA A100-SXM4-80GB


## 3. 內建測試、schema 檢查與真實前兩列轉換

In [12]:
# 先測試 converter 本身。
subprocess.run(
    ['python', 'smoke_test_combined_csv_conversion.py'],
    cwd=REPO_ROOT,
    check=True,
)

# 直接由 streaming converter 檢查真實 CSV 前兩列。
inspect = subprocess.run(
    [
        'python', 'combined_csv_to_ssif_json.py', 'inspect',
        '--csv', str(DATA_CSV),
        '--rows', '2',
        '--horizon', '120',
    ],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(inspect.stdout)
if inspect.stderr:
    print(inspect.stderr)
assert inspect.returncode == 0
inspect_summary = json.loads(inspect.stdout)
assert inspect_summary['layout'] == 'event_json'
assert len(inspect_summary['sample_rows']) == 2

sample_csv = Path('/content/combined_data_sample_2rows.csv')
sample_out = Path('/content/ssif_combined_sample_events')
shutil.rmtree(sample_out, ignore_errors=True)

# 重要：field_size_limit 已在上一格提高。
with DATA_CSV.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    fieldnames = list(reader.fieldnames or [])
    rows = []
    for index, row in enumerate(reader):
        rows.append(row)
        if index == 1:
            break

assert len(rows) == 2
assert {'eq_info', 'intensity'}.issubset(fieldnames)

with sample_csv.open('w', encoding='utf-8', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

subprocess.run(
    [
        'python', 'combined_csv_to_ssif_json.py', 'convert',
        '--csv', str(sample_csv),
        '--output-dir', str(sample_out),
        '--horizon', '120',
        '--duplicate-policy', 'skip-identical',
        '--error-policy', 'error',
        '--overwrite',
    ],
    cwd=REPO_ROOT,
    check=True,
)

subprocess.run(
    [
        'python', 'combined_csv_to_ssif_json.py', 'validate',
        '--data-dir', str(sample_out),
        '--horizon', '120',
    ],
    cwd=REPO_ROOT,
    check=True,
)

sample_files = sorted(sample_out.glob('event_*.json'))
assert len(sample_files) == 2
sample_event = json.loads(sample_files[0].read_text(encoding='utf-8'))
first_station = next(iter(sample_event['intensity']))
assert len(sample_event['intensity'][first_station]) == 120
print('PASS: real two-row conversion')


{
  "csv_path": "/content/drive/MyDrive/00_SSIF/combined_data.csv",
  "file_size_bytes": 727934696,
  "columns": [
    "times",
    "stids",
    "intensity",
    "epicenter_distance",
    "variables",
    "eq_info",
    "source_file"
  ],
  "layout": "event_json",
  "sample_rows": [
    {
      "row": 1,
      "identity": "2024-02-20T14:57:31|123.2540|23.7213|58.9|5.7",
      "origin_time": "2024-02-20T14:57:31",
      "stations": 546,
      "short_sequences": 0,
      "long_sequences": 0
    },
    {
      "row": 2,
      "identity": "2024-04-03T00:00:19|121.5850|23.8075|30.7|5.7",
      "origin_time": "2024-04-03T00:00:19",
      "stations": 535,
      "short_sequences": 0,
      "long_sequences": 0
    }
  ]
}

PASS: real two-row conversion


## 4. 完整 scan、轉換與驗證

In [13]:
if RUN_FULL_SCAN:
    subprocess.run(
        [
            'python', 'combined_csv_to_ssif_json.py', 'scan',
            '--csv', str(DATA_CSV),
            '--horizon', '120',
            '--max-errors', '100',
            '--report', str(SCAN_REPORT),
        ],
        cwd=REPO_ROOT,
        check=True,
    )

scan_summary = (
    json.loads(SCAN_REPORT.read_text(encoding='utf-8'))
    if SCAN_REPORT.is_file() else None
)

if scan_summary:
    display(pd.DataFrame.from_dict(
        scan_summary['counters'],
        orient='index',
        columns=['count'],
    ))
    if scan_summary['errors']:
        display(pd.DataFrame(scan_summary['errors']))
    if scan_summary['duplicate_examples']:
        display(pd.DataFrame(scan_summary['duplicate_examples']))

if RUN_FULL_CONVERSION:
    assert scan_summary is not None, '請先執行完整 scan'
    counters = scan_summary['counters']
    assert counters.get('row_errors', 0) == 0
    if counters.get('identity_conflicts', 0):
        assert DUPLICATE_POLICY in {'merge', 'suffix'}

    process = subprocess.run(
        [
            'python', 'combined_csv_to_ssif_json.py', 'convert',
            '--csv', str(DATA_CSV),
            '--output-dir', str(TRAIN_DATA),
            '--horizon', '120',
            '--duplicate-policy', DUPLICATE_POLICY,
            '--error-policy', 'error',
            '--max-errors', '100',
            '--overwrite',
        ],
        cwd=REPO_ROOT,
        text=True,
)
    if process.returncode:
        report = TRAIN_DATA / 'conversion_error_report.json'
        if report.is_file():
            print(report.read_text(encoding='utf-8'))
        raise RuntimeError('完整轉換失敗')

if RUN_FULL_VALIDATION:
    subprocess.run(
        [
            'python', 'combined_csv_to_ssif_json.py', 'validate',
            '--data-dir', str(TRAIN_DATA),
            '--horizon', '120',
            '--max-errors', '100',
            '--report', str(VALIDATION_REPORT),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
    validation = json.loads(VALIDATION_REPORT.read_text(encoding='utf-8'))
    assert validation['counters'].get('errors', 0) == 0
    print('PASS: full archive validation')


## 5. 資料切分與模型訓練

In [14]:
if RUN_AUDIT_SPLIT:
    assert not any(PREPARED_DIR.iterdir()), (
        'PREPARED_DIR 已有輸出；新資料版本請改用 split_v2'
    )
    subprocess.run(
        [
            'python', 'prepare_ssif_dataset.py', 'audit-split',
            '--data-dir', str(TRAIN_DATA),
            '--output-dir', str(PREPARED_DIR),
            '--windows', *map(str, WINDOWS),
            '--label-horizon', '120',
            '--min-label-valid-fraction', '0.80',
            '--min-window-valid-fraction', '0.80',
            '--train-ratio', '0.70',
            '--validation-ratio', '0.10',
            '--calibration-ratio', '0.10',
            '--test-ratio', '0.10',
            '--split-candidates', '5000',
            '--seed', str(SEED),
        ],
        cwd=REPO_ROOT,
        check=True,
    )

quick_dir = WORK_ROOT / 'models' / 'quick_EW10'

if RUN_QUICK_TRAIN:
    command = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(quick_dir),
        '--windows', '10',
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '1',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
    ]
    if torch.cuda.is_available():
        command.append('--amp')
    subprocess.run(command, cwd=REPO_ROOT, check=True)

if RUN_FULL_TRAIN:
    command = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(MODEL_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '30',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--weight-decay', '1e-2',
        '--min-precision', '0.90',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--patience', '6',
    ]
    if torch.cuda.is_available():
        command.append('--amp')
    subprocess.run(command, cwd=REPO_ROOT, check=True)


## 執行順序

1. 執行第 1–3 節；應看到 `CSV field limit` 大於 131072，並顯示 `PASS: real two-row conversion`。
2. 將 `RUN_FULL_SCAN=True`，檢查 `row_errors` 與 `identity_conflicts`。
3. scan 通過後將 `RUN_FULL_CONVERSION=True`。
4. 轉換完成後將 `RUN_FULL_VALIDATION=True`。
5. 再依序開啟 audit、quick train 與 full train。